# Set 04 – Daten verstehen und visualisieren

Bevor ein Algorithmus zum Einsatz kommt, müssen wir die Daten kennenlernen. Dieses Notebook führt die wichtigsten Pandas- und Visualisierungsfunktionen einzeln ein. Die Daten sind kontrolliert erzeugt, damit fehlende Werte, Ausreißer und Zusammenhänge gut sichtbar sind.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
n = 240
alter = np.clip(rng.normal(42, 13, n), 18, 78).round()
dauer = rng.integers(1, 73, n)
vertrag = rng.choice(["Monatlich", "Jährlich", "Zwei Jahre"], n, p=[0.48, 0.34, 0.18])
region = rng.choice(["Nord", "Süd", "West", "Ost"], n)
ausgaben = 22 + 0.55 * alter + 0.75 * dauer + rng.normal(0, 12, n)
zufriedenheit = np.clip(2.2 + ausgaben / 65 + rng.normal(0, 0.65, n), 1, 5)

daten = pd.DataFrame({
    "alter": alter,
    "vertragsdauer_monate": dauer,
    "monatliche_ausgaben": ausgaben.round(2),
    "zufriedenheit": zufriedenheit.round(1),
    "vertrag": vertrag,
    "region": region,
})
daten.loc[rng.choice(n, 12, replace=False), "zufriedenheit"] = np.nan
daten.loc[rng.choice(n, 8, replace=False), "region"] = np.nan
daten.loc[[7, 133], "monatliche_ausgaben"] = [185, 205]

## 1. Einen ersten Überblick bekommen

- head(n) zeigt die ersten Zeilen.
- tail(n) zeigt die letzten Zeilen.
- sample(n) zieht zufällige Zeilen.
- shape liefert Zeilen und Spalten.
- columns enthält die Spaltennamen.
- dtypes zeigt den Datentyp jeder Spalte.

In [ ]:
display(daten.head(3))
display(daten.tail(3))
display(daten.sample(3, random_state=1))
print("Form:", daten.shape)
print("Spalten:", daten.columns.tolist())
print("Datentypen:")
print(daten.dtypes)

### info() – kompakte Strukturprüfung

info() zeigt Datentypen, Speicherbedarf und wie viele Werte pro Spalte nicht fehlen.

In [ ]:
daten.info()

## 2. Fehlende Werte finden

isna() markiert fehlende Werte mit True. Durch anschließendes sum() werden sie spaltenweise gezählt.

In [ ]:
fehlend = daten.isna().sum().sort_values(ascending=False)
display(fehlend.to_frame("anzahl_fehlend"))

## 3. Numerische Spalten beschreiben

select_dtypes(include="number") wählt Zahlenspalten aus. describe() berechnet Mittelwert, Standardabweichung, Minimum, Quartile und Maximum. Ein großer Abstand zwischen dem 75%-Quartil und Maximum kann auf Ausreißer hinweisen.

In [ ]:
numerisch = daten.select_dtypes(include="number")
display(numerisch.describe().T)

## 4. Kategorien untersuchen

value_counts() zählt Ausprägungen. Mit normalize=True erhalten wir Anteile; dropna=False schließt fehlende Werte ein. nunique() zählt unterschiedliche Werte.

In [ ]:
print("Unterschiedliche Werte pro Spalte:")
print(daten.nunique(dropna=False))
display(daten["vertrag"].value_counts(dropna=False).to_frame("anzahl"))
display(daten["region"].value_counts(normalize=True, dropna=False).rename("anteil").to_frame())

## 5. Verteilungen mit Histogrammen

Ein Histogramm teilt einen Zahlenbereich in Intervalle und zählt die Beobachtungen darin. So werden Lage, Streuung und Schiefe sichtbar.

In [ ]:
daten[["alter", "vertragsdauer_monate", "monatliche_ausgaben"]].hist(
    bins=20, figsize=(12, 3.5), color="#4C78A8", edgecolor="white"
)
plt.suptitle("Verteilungen numerischer Merkmale", y=1.04)
plt.tight_layout()
plt.show()

## 6. Ausreißer mit dem Boxplot erkennen

Die Box reicht vom ersten bis zum dritten Quartil, die Linie darin ist der Median. Punkte außerhalb der Whisker sind auffällig, aber nicht automatisch fehlerhaft.

In [ ]:
ax = daten.boxplot(column="monatliche_ausgaben", vert=False, figsize=(9, 3.5))
ax.set_title("Monatliche Ausgaben – zwei kontrollierte Ausreißer")
ax.set_xlabel("monatliche Ausgaben")
plt.show()

## 7. Scatterplot und Gruppenvergleich

scatterplot() zeigt zwei numerische Merkmale gegeneinander. groupby() fasst Zeilen nach Kategorien zusammen. Der Median ist gegenüber Ausreißern robuster als der Mittelwert.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, gruppe in daten.groupby("vertrag", observed=True):
    axes[0].scatter(gruppe["vertragsdauer_monate"], gruppe["monatliche_ausgaben"], label=name, alpha=0.7)
axes[0].set_xlabel("Vertragsdauer in Monaten")
axes[0].set_ylabel("monatliche Ausgaben")
axes[0].legend(title="Vertrag")
axes[0].set_title("Zwei numerische Merkmale")
gruppen = daten.groupby("vertrag", observed=True)["monatliche_ausgaben"].median().sort_values()
gruppen.plot.bar(ax=axes[1], color="#54A24B")
axes[1].set_title("Median nach Vertragsart")
axes[1].set_ylabel("monatliche Ausgaben")
plt.tight_layout()
plt.show()

## 8. Korrelationen

corr() misst lineare Zusammenhänge zwischen numerischen Spalten. Werte liegen zwischen -1 und 1. Korrelation ist kein Beweis für Ursache und Wirkung.

In [ ]:
korrelation = numerisch.corr()
fig, ax = plt.subplots(figsize=(7, 5))
bild = ax.imshow(korrelation, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(korrelation.columns)), korrelation.columns, rotation=45, ha="right")
ax.set_yticks(range(len(korrelation.index)), korrelation.index)
for zeile in range(len(korrelation.index)):
    for spalte in range(len(korrelation.columns)):
        ax.text(spalte, zeile, f"{korrelation.iloc[zeile, spalte]:.2f}", ha="center", va="center")
fig.colorbar(bild, ax=ax, label="Korrelation")
ax.set_title("Korrelationsmatrix")
plt.tight_layout()
plt.show()

## Checkliste vor der Vorverarbeitung

1. Was bedeutet eine Zeile und was bedeutet jede Spalte?
2. Stimmen Form, Namen und Datentypen?
3. Wo und warum fehlen Werte?
4. Sind Kategorien plausibel und konsistent geschrieben?
5. Gibt es Ausreißer oder unmögliche Werte?
6. Welche Verteilungen und Zusammenhänge sind sichtbar?

Erst danach entscheiden wir, welche Vorverarbeitung sinnvoll ist.